In [22]:
# ==========================================================
# TEMPORAL FUSION TRANSFORMER (TFT)
# HYPERPARAMETER TUNING PIPELINE
# ==========================================================

# pip install pytorch-lightning pytorch-forecasting

import pandas as pd
import numpy as np
import itertools

from sklearn.model_selection import TimeSeriesSplit

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer
)

from pytorch_forecasting.metrics import QuantileLoss

from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping

import torch

# ==========================================================
# 1. PREPARE TFT DATAFRAME
# ==========================================================

def prepare_tft_dataframe(ts):

    df = ts.copy()

    df = df.reset_index()

    df = df.rename(columns={
        'date': 'time',
        'flow': 'value'
    })

    # time index
    df["time_idx"] = np.arange(len(df))

    # single-series group
    df["series"] = "river"

    # month feature
    df["month"] = df["time"].dt.month.astype(str)

    return df


# ==========================================================
# 2. NSE
# ==========================================================

def nse(y_true, y_pred):

    denom = np.sum(
        (y_true - np.mean(y_true))**2
    )

    if denom == 0:
        return 0

    return 1 - np.sum(
        (y_true - y_pred)**2
    ) / denom


# -------------------------------------------------
# PREPARE TIME SERIES
# -------------------------------------------------
def prepare_time_series(df, value_name='flow'):

    df = df.copy()

    # clean column names
    df.columns = df.columns.str.strip().str.lower()

    months = [
        "jan","feb","mar","apr","maj","jun",
        "jul","avg","sep","okt","nov","dec"
    ]

    existing_months = [m for m in months if m in df.columns]

    # convert numeric
    df[existing_months] = (
        df[existing_months]
        .apply(pd.to_numeric, errors='coerce')
    )

    # fill missing values
    df[existing_months] = (
        df[existing_months]
        .ffill()
        .bfill()
    )

    # long format
    ts = df.melt(
        id_vars='year',
        value_vars=existing_months,
        var_name='month',
        value_name=value_name
    )

    # datetime index
    ts['date'] = pd.to_datetime(
        ts['year'].astype(str) + '-' + ts['month'],
        format='%Y-%b',
        errors='coerce'
    )

    ts = ts.dropna(subset=['date'])

    ts = ts.set_index('date').sort_index()

    ts = ts.dropna(subset=[value_name])

    # non-negative
    ts[value_name] = ts[value_name].clip(lower=0)

    return ts


In [23]:
# ==========================================================
# 3. TFT TUNING
# ==========================================================
from pytorch_forecasting.metrics import QuantileLoss


def tune_tft(ts,
             encoder_length=12,
             prediction_length=1):

    df = prepare_tft_dataframe(ts)

    # ------------------------------------------------------
    # TRAIN/VALIDATION SPLIT
    # ------------------------------------------------------

    split_idx = df[df["time"] < "2014-01-01"].index[-1]

    train_df = df.iloc[:split_idx]
    val_df   = df.iloc[split_idx:]

    # ------------------------------------------------------
    # DATASET
    # ------------------------------------------------------

    training = TimeSeriesDataSet(

        train_df,

        time_idx="time_idx",

        target="value",

        group_ids=["series"],

        max_encoder_length=encoder_length,

        max_prediction_length=prediction_length,

        static_categoricals=["series"],

        time_varying_known_categoricals=["month"],

        time_varying_known_reals=["time_idx"],

        time_varying_unknown_reals=["value"],

        target_normalizer=None
    )

    validation = TimeSeriesDataSet.from_dataset(
        training,
        val_df,
        predict=True,
        stop_randomization=True
    )

    train_loader = training.to_dataloader(
        train=True,
        batch_size=32
    )

    val_loader = validation.to_dataloader(
        train=False,
        batch_size=32
    )

    # ------------------------------------------------------
    # PARAMETER GRID
    # ------------------------------------------------------

    param_grid = {

        'hidden_size': [8, 16, 32],

        'attention_head_size': [1, 2, 4],

        'dropout': [0.1, 0.2, 0.3],

        'learning_rate': [0.001, 0.0005]
    }

    best_score = -np.inf
    best_model = None
    best_params = None

    # ------------------------------------------------------
    # GRID SEARCH
    # ------------------------------------------------------

    for hidden_size, head_size, dropout, lr in itertools.product(

        param_grid['hidden_size'],
        param_grid['attention_head_size'],
        param_grid['dropout'],
        param_grid['learning_rate']
    ):

        print("\nTesting:",
              hidden_size,
              head_size,
              dropout,
              lr)

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=20,
            mode="min"
        )

        trainer = Trainer(

            max_epochs=100,

            accelerator="cpu",

            enable_checkpointing=False,

            logger=False,

            callbacks=[early_stop]
        )

        tft = TemporalFusionTransformer.from_dataset(
            training,
            learning_rate=lr,
            hidden_size=hidden_size,
            attention_head_size=head_size,
            dropout=dropout,
            hidden_continuous_size=16,
            output_size=1,   # single prediction
            loss=QuantileLoss(quantiles=[0.5]),  # FIX
            reduce_on_plateau_patience=5
        )

        trainer.fit(
            tft,
            train_loader,
            val_loader
        )

        # --------------------------------------------------
        # VALIDATION PREDICTION
        # --------------------------------------------------

        preds = tft.predict(val_loader)

        y_pred = preds.numpy().flatten()

        y_true = val_df["value"].values[-len(y_pred):]

        # enforce non-negative
        y_pred = np.maximum(y_pred, 0)

        score = nse(y_true, y_pred)

        print("Validation NSE:", score)

        if score > best_score:

            best_score = score

            best_model = tft

            best_params = {

                'hidden_size': hidden_size,

                'attention_head_size': head_size,

                'dropout': dropout,

                'learning_rate': lr
            }

    print("\n====================================")
    print("BEST TFT CONFIGURATION")
    print("====================================")

    print("Best NSE:", best_score)

    print(best_params)

    return best_model, best_params

In [24]:
#!pip install pytorch-forecasting

In [25]:
# PREPARE SERIES
q_min = pd.read_csv("q_min.csv")

ts_qmin = prepare_time_series(q_min)

# TUNE TFT
best_tft_model, best_tft_params = tune_tft(ts_qmin)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



Testing: 8 1 0.1 0.001


/home/marko/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/marko/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │     46 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │     64 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │     24 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  1.1 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │    536 │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │    304 │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │    304 │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │    304 │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │    304 │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │    576 │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │    576 │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    144 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     16 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │    368 │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    280 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    160 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │    304 │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    160 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │      9 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 5.4 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.4 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 199                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │     46 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │     64 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │     96 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  4.0 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  2.1 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     33 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 59.0 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 59.0 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 205                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Validation NSE: 0

BEST TFT CONFIGURATION
Best NSE: 0
{'hidden_size': 8, 'attention_head_size': 1, 'dropout': 0.1, 'learning_rate': 0.001}
